# Projeto 2ª VA: comparação de algoritmos de classificação

Base usada: Sleep Health and Lifestyle (a mesma da Entrega 1, já tratada).

O que queremos: descobrir se a pessoa não tem distúrbio do sono, se tem insônia ou se tem apneia.

O que vamos fazer neste notebook:
1. testar 6 algoritmos: Árvore de Decisão, KNN, Naive Bayes, Regressão Logística, SVM e MLP
2. avaliar cada um com validação cruzada de 10 folds (estratificada)
3. testar várias combinações de parâmetros com o GridSearchCV
4. mostrar a média e o desvio padrão dos resultados nos 10 folds.

In [1]:
# Instala as bibliotecas (só precisa rodar uma vez)
%pip install -q pandas numpy matplotlib scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    make_scorer, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)

# Desliga os avisos (warnings) para a saída ficar mais limpa
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

SEED = 42        # semente fixa, para o resultado ser sempre o mesmo
N_FOLDS = 10     # quantidade de folds da validação cruzada

# No arquivo tratado as classes viraram números: 0 = None, 1 = Insomnia, 2 = Sleep Apnea
NOMES_CLASSES = ["None", "Insomnia", "Sleep Apnea"]

# Pasta onde vamos salvar as tabelas e os gráficos
os.makedirs("resultados", exist_ok=True)

## 1. Carregando os dados

Aqui usamos a base que já foi tratada na Entrega 1 (`dados/sleep_preprocessado.csv`). A base original continua em `dados/sleep_original.csv`.

Na Entrega 1 ja havia sido feito a retirada da coluna de ID, juntado rótulos repetidos, separado a pressão arterial em duas colunas e transformado os textos em números. A base não tem valores faltando.

In [3]:
df = pd.read_csv("dados/sleep_preprocessado.csv")
alvo = "Sleep Disorder"   # coluna que queremos prever

print("Linhas e colunas:", df.shape)
print("\nQuantidade de pessoas em cada classe:")
print(df[alvo].value_counts().sort_index().rename(index=dict(enumerate(NOMES_CLASSES))))

Linhas e colunas: (374, 20)

Quantidade de pessoas em cada classe:
Sleep Disorder
None           219
Insomnia        77
Sleep Apnea     78
Name: count, dtype: int64


### Linhas repetidas

Na Entrega 1 vimos que 242 das 374 linhas são repetidas (são cópias de outras linhas).

Isso é um problema na validação cruzada: uma cópia pode ir para o treino e outra igualzinha para o teste. Aí o modelo "já viu a resposta" e o resultado parece melhor do que é.

Como o enunciado não fala nada sobre isso, rodamos tudo com as duas versões da base:
- completa: as 374 linhas
- sem_duplicatas: só as 132 linhas diferentes (cada classe ainda tem mais de 10 linhas, então ainda dá para fazer 10 folds).

In [4]:
# drop_duplicates() apaga as linhas repetidas e deixa só uma de cada
df_unico = df.drop_duplicates().reset_index(drop=True)
df_unico.to_csv("dados/sleep_preprocessado_sem_duplicatas.csv", index=False)

# Guardamos as duas versões num dicionário para rodar o mesmo código nas duas
bases = {"completa": df, "sem_duplicatas": df_unico}

for nome, d in bases.items():
    print(f"{nome}: {d.shape[0]} linhas | por classe:", d[alvo].value_counts().sort_index().tolist())

completa: 374 linhas | por classe: [219, 77, 78]
sem_duplicatas: 132 linhas | por classe: [73, 29, 30]
